## Setup Catalog and Schemas

In [0]:
%run ./00_config

In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("environment")

config = get_config(env)
print_config(config)

In [0]:
catalog_name = config["catalog_name"]

schemas = [
    config["raw_schema"],
    config["bronze_schema"],
    config["silver_schema"],
    config["gold_schema"],
    config["governance_schema"],
    config["quarantine_schema"]
]

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

for schema_name in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {catalog_name}.{config['raw_schema']}.{config['source_volume']}
""")

print("Catalog, schemas, and source volume created successfully.")

In [0]:
dq_table = build_table_name(config, "governance_schema", "data_quality_results")
audit_table = build_table_name(config, "governance_schema", "pipeline_audit_log")
classification_table = build_table_name(config, "governance_schema", "data_classification")
lineage_table = build_table_name(config, "governance_schema", "data_lineage")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {dq_table} (
    dq_run_id STRING,
    table_name STRING,
    rule_name STRING,
    rule_type STRING,
    validation_status STRING,
    failed_record_count BIGINT,
    severity STRING,
    execution_datetime TIMESTAMP,
    error_message STRING
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_table} (
    pipeline_run_id STRING,
    notebook_name STRING,
    layer_name STRING,
    source_table STRING,
    target_table STRING,
    records_read BIGINT,
    records_written BIGINT,
    records_rejected BIGINT,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    error_message STRING
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {classification_table} (
    catalog_name STRING,
    schema_name STRING,
    table_name STRING,
    column_name STRING,
    classification_type STRING,
    sensitivity_level STRING,
    masking_required BOOLEAN,
    business_owner STRING
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {lineage_table} (
    source_table STRING,
    target_table STRING,
    transformation_name STRING,
    notebook_name STRING,
    layer_from STRING,
    layer_to STRING,
    created_datetime TIMESTAMP
)
""")

print("Governance tables created successfully.")


In [0]:
spark.sql(f"SHOW SCHEMAS IN {catalog_name}").show(truncate=False)

In [0]:
print("Upload CSV files to this volume path:")
print(build_volume_path(config))